# Quantum Hello World

This notebook builds one small thing, a random number generator, in six steps. By the end you will know what a quantum circuit is made of, how to build one in Qiskit, how to run it, and how to read the result.

**Why a random number generator?**

Real randomness is valuable. Encryption keys, lotteries, and large simulations all need numbers nobody can predict. A classical computer cannot make them. It can only fake them with arithmetic that looks random but is fully determined by a starting value. A quantum computer produces randomness from physics, not arithmetic. Reading a qubit is random in a way that no formula reproduces.

**One honest note before we start.** Today we run on a simulator. The simulator is a normal program that pretends to be a quantum computer, so its randomness is faked like any other program. The circuits you write here are exactly the circuits that run on real hardware. Only the last line, where you choose what runs them, would change.

**How to use this notebook.** Run each cell in order with Shift+Enter. Every section ends with an exercise cell. Change the value it asks you to change, predict what will happen, then run it.


## Setup

Qiskit is the Python library for writing quantum circuits. The `[visualization]` extra adds the pieces that draw circuits and histograms. `qiskit-aer` is the simulator. This cell takes about a minute the first time.

In [ ]:
!pip install -q "qiskit[visualization]" qiskit-aer

Every setting used below lives in this one cell. Nothing is hardcoded later.

In [ ]:
# Number of times each circuit is run. One run is called a "shot".
SHOTS = 1000

# Number of qubits in the random byte in Section 5.
BITS_IN_RANDOM_NUMBER = 8

# Number of dice rolls to collect in Section 5.
DICE_ROLLS = 1000

In [ ]:
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator

simulator = AerSimulator()
print("Setup complete.")

## 1. The classical baseline

Before writing any quantum code, look at what we are trying to match. Python's `random` module gives a 0 or a 1 with equal probability. If we ask for 1000 of them and count, we expect about 500 of each.

This histogram is the picture to keep in mind. In the next section a quantum circuit produces the same picture from a different source.

In [ ]:
classical_bits = [random.randint(0, 1) for _ in range(SHOTS)]

# Counter turns the list into a dictionary: {0: how many zeros, 1: how many ones}
classical_counts = Counter(classical_bits)
print(classical_counts)

# plot_histogram wants string labels, which is how Qiskit reports results later.
plot_histogram({str(bit): count for bit, count in classical_counts.items()})

## 2. One qubit, one random bit

A quantum circuit has three kinds of parts:

1. **Qubits.** The things that hold information. A qubit always starts at 0.
2. **Gates.** Operations applied to qubits, drawn as boxes on a line. They change what the qubit will give you when read.
3. **Measurements.** Reading a qubit. Measuring always gives a plain 0 or 1, and that bit is written into a classical bit so you can see it.

The gate we need is called **H**, for Hadamard. Applied to a qubit at 0, it puts the qubit into a state where measuring gives 0 half the time and 1 half the time. That is the whole random bit generator: one qubit, one H gate, one measurement.

Running a circuit once gives one bit. The simulator runs it `SHOTS` times and counts the results.

In the code, `QuantumCircuit(1, 1)` means one qubit and one classical bit. `measure(0, 0)` means read qubit 0 into classical bit 0.

In [ ]:
random_bit_circuit = QuantumCircuit(1, 1)
random_bit_circuit.h(0)
random_bit_circuit.measure(0, 0)

# Draw the circuit. Time runs left to right. The top line is the qubit, the double line is the classical bit.
random_bit_circuit.draw("mpl")

In [ ]:
random_bit_result = simulator.run(random_bit_circuit, shots=SHOTS).result()
random_bit_counts = random_bit_result.get_counts()
print(random_bit_counts)
plot_histogram(random_bit_counts)

Same picture as Section 1. The difference is where it came from. Python computed its bits with arithmetic. The circuit read them from a qubit.

**Exercise 2.** Remove the H gate by changing `USE_H_GATE` to `False`. Before you run it, predict the histogram. Then put the gate back and change `shots` to 10. How uneven can 10 shots be?

In [ ]:
USE_H_GATE = True

exercise_circuit = QuantumCircuit(1, 1)
if USE_H_GATE:
    exercise_circuit.h(0)
exercise_circuit.measure(0, 0)

exercise_counts = simulator.run(exercise_circuit, shots=SHOTS).result().get_counts()
print(exercise_counts)
plot_histogram(exercise_counts)

## 3. Look before you measure

On a real quantum computer you can never see a qubit's state. You can only measure it and get a 0 or a 1. The simulator has no such rule. It will show you the state directly, and that is worth one look.

A one-qubit state is two numbers, one attached to the outcome 0 and one attached to the outcome 1. They are called **amplitudes**. The probability of an outcome is the square of its amplitude.

After an H gate both amplitudes are 0.707, and 0.707 squared is 0.5. That is where the fifty-fifty came from.

To look at the state we build the circuit without a measurement, because measuring is what forces the state to become a plain bit.

In [ ]:
unmeasured_circuit = QuantumCircuit(1)
unmeasured_circuit.h(0)

state_after_h = Statevector(unmeasured_circuit)

# Amplitudes are complex numbers in general. None of today's gates produce an imaginary part, so we print only the real part.
amplitudes = np.round(state_after_h.data.real, 3)
print("Amplitude for outcome 0:", amplitudes[0])
print("Amplitude for outcome 1:", amplitudes[1])
print("Probability of 0 (amplitude squared):", round(float(amplitudes[0] ** 2), 3))
print("Probability of 1 (amplitude squared):", round(float(amplitudes[1] ** 2), 3))

Now the part that makes quantum computing different from flipping a switch.

Amplitudes can be **negative**. Probabilities cannot. Apply H a second time and the two amplitudes attached to outcome 1 cancel each other out exactly. The qubit goes back to a certain 0.

Think about what that means. One H gate made the result random. A second H gate undid it. Randomness that can be undone is not something ordinary probability can do. Every quantum algorithm that beats a classical one works by arranging for wrong answers to cancel like this and right answers to add up.

In [ ]:
double_h_circuit = QuantumCircuit(1, 1)
double_h_circuit.h(0)
double_h_circuit.h(0)
double_h_circuit.measure(0, 0)

double_h_counts = simulator.run(double_h_circuit, shots=SHOTS).result().get_counts()
print(double_h_counts)
double_h_circuit.draw("mpl")

**Exercise 3.** Change `NUMBER_OF_H_GATES` to 3, then 4. Predict the counts before each run. What is the rule?

In [ ]:
NUMBER_OF_H_GATES = 3

repeated_h_circuit = QuantumCircuit(1, 1)
for _ in range(NUMBER_OF_H_GATES):
    repeated_h_circuit.h(0)
repeated_h_circuit.measure(0, 0)

repeated_h_counts = simulator.run(repeated_h_circuit, shots=SHOTS).result().get_counts()
print(repeated_h_counts)

## 4. A biased bit

H always gives fifty-fifty. A different gate, **RY**, takes an angle and lets you choose the odds.

- Angle 0 leaves the qubit at 0. Measuring gives 0 every time.
- Angle π (about 3.14) flips it fully. Measuring gives 1 every time.
- Angle π/2 is fifty-fifty, the same as H.
- Angles in between give odds in between.

The exact rule is that the probability of measuring 1 is sin²(angle / 2). You do not need to derive that. The next cell checks it. We run the circuit at several angles and plot how often 1 comes up against what the formula predicts.

In [ ]:
angles = np.linspace(0, np.pi, 13)
measured_probability_of_one = []

for angle in angles:
    biased_bit_circuit = QuantumCircuit(1, 1)
    biased_bit_circuit.ry(angle, 0)
    biased_bit_circuit.measure(0, 0)
    biased_counts = simulator.run(biased_bit_circuit, shots=SHOTS).result().get_counts()
    # .get("1", 0) handles angles where 1 never appears in the counts at all.
    measured_probability_of_one.append(biased_counts.get("1", 0) / SHOTS)

predicted_probability_of_one = np.sin(angles / 2) ** 2

plt.plot(angles, predicted_probability_of_one, label="formula: sin^2(angle / 2)")
plt.plot(angles, measured_probability_of_one, "o", label=f"measured, {SHOTS} shots each")
plt.xlabel("RY angle (radians)")
plt.ylabel("fraction of shots that gave 1")
plt.legend()
plt.show()

The dots sit on the curve, with small wobbles from only running 1000 shots. This is how every gate works: it changes the amplitudes by a fixed rule, and the amplitudes set the odds.

**Exercise 4.** Find the angle that gives 1 about 25% of the time. Set `TARGET_ANGLE` and run. You can solve sin²(angle / 2) = 0.25 or just guess and check.

In [ ]:
TARGET_ANGLE = 1.0

target_circuit = QuantumCircuit(1, 1)
target_circuit.ry(TARGET_ANGLE, 0)
target_circuit.measure(0, 0)

target_counts = simulator.run(target_circuit, shots=SHOTS).result().get_counts()
print(target_counts)
print("Fraction of ones:", target_counts.get("1", 0) / SHOTS)

## 5. From bits to numbers

One random bit is not much use. Eight random bits make a byte, a number from 0 to 255. The circuit is the same idea with more qubits: put H on every qubit, measure every qubit.

`QuantumCircuit(8, 8)` makes eight qubits and eight classical bits. `measure_all()` reads each qubit into its own classical bit. Each shot now returns an eight-character string of 0s and 1s, and `int(bits, 2)` turns that string into a number.

We ask the simulator for `memory=True` so it keeps every shot instead of just counting them.

In [ ]:
random_number_circuit = QuantumCircuit(BITS_IN_RANDOM_NUMBER)
for qubit in range(BITS_IN_RANDOM_NUMBER):
    random_number_circuit.h(qubit)
random_number_circuit.measure_all()

random_number_result = simulator.run(random_number_circuit, shots=10, memory=True).result()
random_bitstrings = random_number_result.get_memory()

for bitstring in random_bitstrings:
    print(bitstring, "=", int(bitstring, 2))

random_number_circuit.draw("mpl")

Now something with a reason to exist: **a quantum die.**

Three qubits give a number from 0 to 7. A die needs 1 to 6. The fix is the same one used by every real random number library. If the number is 6 or 7, throw it away and roll again. The numbers 0 through 5 that survive are each equally likely, and adding 1 turns them into 1 through 6.

Throwing away results is called rejection sampling. It wastes a quarter of the shots here, which is fine. The histogram should come out flat.

In [ ]:
die_circuit = QuantumCircuit(3)
die_circuit.h(0)
die_circuit.h(1)
die_circuit.h(2)
die_circuit.measure_all()

# Ask for extra shots because about a quarter of them will be rejected.
die_result = simulator.run(die_circuit, shots=DICE_ROLLS * 2, memory=True).result()
raw_values = [int(bitstring, 2) for bitstring in die_result.get_memory()]

die_rolls = []
for value in raw_values:
    if value <= 5:
        die_rolls.append(value + 1)
    if len(die_rolls) == DICE_ROLLS:
        break

assert len(die_rolls) == DICE_ROLLS, "Not enough shots survived rejection. Raise the shot count."

die_counts = Counter(die_rolls)
print(sorted(die_counts.items()))
plot_histogram({str(face): count for face, count in die_counts.items()})

**Exercise 5.** Make a twenty-sided die. You need 5 qubits (values 0 to 31) and you must reject anything 20 or above. Fill in `NUMBER_OF_QUBITS` and `LARGEST_ALLOWED_VALUE`.

In [ ]:
NUMBER_OF_QUBITS = 3
LARGEST_ALLOWED_VALUE = 5

custom_die_circuit = QuantumCircuit(NUMBER_OF_QUBITS)
for qubit in range(NUMBER_OF_QUBITS):
    custom_die_circuit.h(qubit)
custom_die_circuit.measure_all()

custom_die_result = simulator.run(custom_die_circuit, shots=DICE_ROLLS * 2, memory=True).result()
custom_raw_values = [int(bitstring, 2) for bitstring in custom_die_result.get_memory()]

custom_rolls = []
for value in custom_raw_values:
    if value <= LARGEST_ALLOWED_VALUE:
        custom_rolls.append(value + 1)

custom_counts = Counter(custom_rolls)
print(sorted(custom_counts.items()))
plot_histogram({str(face): count for face, count in custom_counts.items()})

## 6. Two bits that always agree

Everything so far could be done with a very expensive random number generator. This last circuit is the one that cannot.

A **CX** gate (controlled-X) acts on two qubits. If the first qubit is 1, it flips the second. If the first qubit is 0, it does nothing. On ordinary bits that is just an if statement.

Put H on qubit 0 first, so qubit 0 is in the fifty-fifty state, and then apply CX from qubit 0 to qubit 1. Measure both. Each qubit on its own is a perfect random bit. But they always agree. You will see `00` and `11` and never `01` or `10`.

This is **entanglement**. The two qubits do not each carry a secret value that was decided in advance and then copied. Showing that no such hidden copy can exist, even in principle, is what the 2022 Nobel Prize in Physics was for. For today, the practical reading is enough: entangled qubits are random individually and locked together as a group. Secure key distribution, quantum error correction, and every quantum algorithm worth running are built on it.

In [ ]:
entangled_circuit = QuantumCircuit(2, 2)
entangled_circuit.h(0)
entangled_circuit.cx(0, 1)
entangled_circuit.measure([0, 1], [0, 1])

entangled_counts = simulator.run(entangled_circuit, shots=SHOTS).result().get_counts()
print(entangled_counts)
entangled_circuit.draw("mpl")

In [ ]:
plot_histogram(entangled_counts)

**Exercise 6.** Change `ADD_X_TO_QUBIT_1` to `True`. The X gate flips a qubit from 0 to 1 and from 1 to 0, with no randomness. Predict which two outcomes you will see now.

In [ ]:
ADD_X_TO_QUBIT_1 = False

exercise_entangled_circuit = QuantumCircuit(2, 2)
exercise_entangled_circuit.h(0)
exercise_entangled_circuit.cx(0, 1)
if ADD_X_TO_QUBIT_1:
    exercise_entangled_circuit.x(1)
exercise_entangled_circuit.measure([0, 1], [0, 1])

exercise_entangled_counts = simulator.run(exercise_entangled_circuit, shots=SHOTS).result().get_counts()
print(exercise_entangled_counts)
plot_histogram(exercise_entangled_counts)

## What you now know

- A circuit is qubits, gates, and measurements. Measuring gives a bit.
- **H** makes a fifty-fifty bit. **RY** makes a bit with any odds you want. **X** flips a bit. **CX** flips one qubit based on another.
- A qubit's state is a list of amplitudes. Probability is amplitude squared. Amplitudes can be negative, so they can cancel. That cancellation is the whole trick.
- More qubits give more bits. Reject what you do not want to turn bits into dice.
- Entangled qubits are random alone and locked together.

**Where this goes next.** The afternoon starter notebooks use the same four moves: build a circuit, choose gates, run it, read the counts. The circuits are bigger and the gates have angles that get tuned by an optimizer, but you have already written every piece.

**Qiskit calls you used today**, for reference:

| Call | What it does |
|---|---|
| `QuantumCircuit(n_qubits, n_classical_bits)` | Make an empty circuit |
| `.h(q)` `.x(q)` `.ry(angle, q)` `.cx(control, target)` | Add a gate |
| `.measure(q, c)` or `.measure_all()` | Read qubits into classical bits |
| `.draw("mpl")` | Draw the circuit |
| `simulator.run(circuit, shots=N).result().get_counts()` | Run it and count outcomes |
| `Statevector(circuit)` | Look at amplitudes (simulator only, no measurements in the circuit) |
